# Calidad de Datos y Estructura General — RT_IOT2022
**Responsable:** Edwin  
**Objetivo:** Revisar la salud general del dataset: tipos de datos, valores faltantes, duplicados, columnas constantes y proponer limpiezas iniciales.

## 1. Carga del dataset

In [ ]:
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

with zipfile.ZipFile('../data/rt-iot2022.zip', 'r') as z:
    with z.open('RT_IOT2022') as f:
        df = pd.read_csv(f, index_col=0)

print(f'Dataset cargado correctamente: {df.shape[0]:,} filas × {df.shape[1]} columnas')

## 2. Dimensiones y primeras filas

In [ ]:
print(f'Filas   : {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]}')
df.head()

## 3. Tipos de datos (df.info)

In [ ]:
df.info(verbose=True, show_counts=True)

## 4. Clasificación de variables: numéricas vs categóricas

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

print(f'Variables NUMÉRICAS ({len(num_cols)}):')
print(num_cols)
print()
print(f'Variables CATEGÓRICAS ({len(cat_cols)}):')
print(cat_cols)

## 5. Valores faltantes

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'nulos': missing, 'porcentaje': missing_pct})
missing_con_nulos = missing_df[missing_df['nulos'] > 0]

if missing_con_nulos.empty:
    print('✔ No hay valores nulos en el dataset.')
else:
    print(f'Se encontraron {len(missing_con_nulos)} columnas con valores nulos:')
    display(missing_con_nulos.sort_values('nulos', ascending=False))

## 6. Filas duplicadas

In [ ]:
n_dup = df.duplicated().sum()
print(f'Filas duplicadas exactas: {n_dup:,} ({n_dup/len(df)*100:.2f}%)')

if n_dup > 0:
    print('Muestra de duplicados:')
    display(df[df.duplicated(keep=False)].sort_values(list(df.columns[:5])).head(10))

## 7. Valores únicos por columna

In [ ]:
unique_counts = df.nunique().sort_values()
unique_df = pd.DataFrame({
    'valores_unicos': unique_counts,
    'porcentaje_del_total': (unique_counts / len(df) * 100).round(4)
})

print('Valores únicos por columna (de menor a mayor):')
display(unique_df)

## 8. Columnas constantes o casi constantes

In [ ]:
UMBRAL_CASI_CONSTANTE = 0.01  # menos del 1% de variación

constantes = []
casi_constantes = []

for col in df.columns:
    n_uniq = df[col].nunique()
    if n_uniq == 1:
        constantes.append(col)
    elif n_uniq / len(df) < UMBRAL_CASI_CONSTANTE:
        valor_dominante = df[col].value_counts(normalize=True).iloc[0]
        if valor_dominante > 0.99:
            casi_constantes.append((col, df[col].value_counts().index[0], round(valor_dominante*100, 2)))

print(f'Columnas CONSTANTES (1 único valor): {constantes if constantes else "Ninguna"}')
print()
print('Columnas CASI CONSTANTES (>99% un solo valor):')
if casi_constantes:
    for col, val, pct in casi_constantes:
        print(f'  {col}: valor dominante = {repr(val)} ({pct}%)')
else:
    print('  Ninguna')

## 9. Columnas con posibles tipos incorrectos

In [ ]:
# Columnas object que en realidad podrían ser numéricas
print('Columnas object — revisión de valores únicos para detectar tipos incorrectos:')
for col in cat_cols:
    sample = df[col].dropna().unique()[:10]
    print(f'\n  {col} ({df[col].nunique()} únicos): {sample}')

# Columnas numéricas que actúan como identificadores (cardinalidad muy alta)
print('\nColumnas numéricas con alta cardinalidad (posibles IDs o puertos):')
for col in num_cols:
    if df[col].nunique() / len(df) > 0.5:
        print(f'  {col}: {df[col].nunique():,} valores únicos')

## 10. Variables principales — análisis detallado

In [ ]:
vars_principales = ['proto', 'service', 'Attack_type', 'id.orig_p', 'id.resp_p']

for col in vars_principales:
    print(f'\n{"="*55}')
    print(f'  {col}  ({df[col].dtype}, {df[col].nunique()} únicos)')
    print('='*55)
    display(df[col].value_counts(dropna=False).to_frame('conteo').assign(
        porcentaje=lambda x: (x['conteo'] / len(df) * 100).round(2)
    ).head(20))

## 11. Índice residual del CSV (`Unnamed: 0`)

In [ ]:
print('Unnamed: 0 / índice residual del CSV:')
print(f'  Nombre del índice: {df.index.name!r}')
print(f'  Rango del índice : {df.index.min()} → {df.index.max()}')
print(f'  Es secuencial    : {list(df.index) == list(range(len(df)))}')
print()
print('→ Al cargar con index_col=0, el índice residual queda como índice del DataFrame.')
print('  No es necesario eliminarlo como columna.')

## 12. Distribución de la variable objetivo `Attack_type`

In [ ]:
counts = df['Attack_type'].value_counts()
pcts   = (counts / len(df) * 100).round(2)

summary = pd.DataFrame({'conteo': counts, 'porcentaje': pcts})
display(summary)

fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette('tab10', len(counts))
bars = ax.barh(counts.index[::-1], counts.values[::-1], color=colors[::-1])

for bar, val in zip(bars, counts.values[::-1]):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

ax.set_xlabel('Número de registros')
ax.set_title('Distribución de Attack_type (RT_IOT2022)')
plt.tight_layout()
plt.show()

print(f'\nClase mayoritaria : {counts.index[0]} ({pcts.iloc[0]}%)')
print(f'Clase minoritaria : {counts.index[-1]} ({pcts.iloc[-1]}%)')
print(f'Ratio desbalance  : {counts.iloc[0] / counts.iloc[-1]:.0f}:1')

## 13. Inconsistencias en nombres de clases o categorías

In [ ]:
for col in cat_cols:
    valores = df[col].dropna().unique()
    tiene_espacios = any(str(v) != str(v).strip() for v in valores)
    tiene_mixto    = len(set(str(v).lower() for v in valores)) < len(set(str(v) for v in valores))

    issues = []
    if tiene_espacios: issues.append('espacios extra')
    if tiene_mixto:    issues.append('mayúsculas inconsistentes')

    if issues:
        print(f'  {col}: {issues}')
    else:
        print(f'  {col}: sin inconsistencias detectadas')

# Revisión manual de Attack_type
print('\nValores de Attack_type:')
for v in sorted(df['Attack_type'].unique()):
    print(f'  {repr(v)}')

## 14. Resumen: primeras limpiezas propuestas

In [ ]:
resumen = {
    '¿Hay valores nulos?'         : 'NO — ninguna columna tiene valores faltantes.',
    '¿Hay filas duplicadas?'       : f'{df.duplicated().sum():,} filas duplicadas exactas.',
    '¿Columnas que no aportan?'   : 'Ninguna — index_col=0 ya maneja el índice residual al cargar.',
    '¿Columnas categóricas?'      : str(cat_cols),
    '¿Columnas numéricas?'        : f'{len(num_cols)} columnas (ver lista en sección 4)',
    '¿Columnas a eliminar?'       : 'Ninguna obligatoria; evaluar id.orig_p e id.resp_p como features'
}

print('RESUMEN DE CALIDAD DE DATOS')
print('='*60)
for pregunta, respuesta in resumen.items():
    print(f'\n{pregunta}')
    print(f'  → {respuesta}')

In [ ]:
print('LIMPIEZAS PROPUESTAS')
print('='*60)
limpiezas = [
    '1. El índice residual se absorbe automáticamente con index_col=0 al cargar desde ZIP.',
    '2. Eliminar filas duplicadas si las hay (df.drop_duplicates()).',
    '3. Convertir `proto` y `service` a tipo category para ahorro de memoria.',
    '4. Evaluar si `id.orig_p` e `id.resp_p` deben usarse como features o solo como identificadores.',
    '5. Revisar columnas casi constantes para posible eliminación en la fase de modelado.',
    '6. Documentar el desbalance fuerte en Attack_type (DOS_SYN_Hping representa ~77% del dataset).',
]
for l in limpiezas:
    print(f'  {l}')

---
*Notebook: Edwin — Calidad de datos y estructura general | Dataset: RT_IOT2022 | 123,117 filas × 84 columnas*